In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
train_df = pd.read_csv('/train.csv')
test_df = pd.read_csv('/test.csv')

In [ ]:
y = np.log1p(train['SalePrice'])
X = train.drop(columns=['Id', 'SalePrice'])
X_test = test.drop(columns=['Id'])

In [ ]:
X = train_df.drop(columns=['Id', 'SalePrice'])
y = train_df['SalePrice']
X_test = test_df.drop(columns=['Id'])

In [ ]:
y_log = np.log1p(y)

In [ ]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

In [ ]:
numerical_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

In [ ]:
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y_log, test_size=0.2, random_state=42)

clf.fit(X_train, y_train)
val_preds = clf.predict(X_val)

rmse = root_mean_squared_error(y_val, val_preds)
print(f"Validation Root Mean Squared Log Error (RMSLE): {rmse:.4f}")

Validation Root Mean Squared Log Error (RMSLE): 0.1458


In [ ]:

print("Training on full dataset and generating predictions...")
clf.fit(X, y_log)
final_preds_log = clf.predict(X_test)
# Convert back from log scale to actual dollars
final_preds = np.expm1(final_preds_log)


Training on full dataset and generating predictions...


In [ ]:
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': final_preds
})
submission.to_csv('my_submission.csv', index=False)
print("Submission saved successfully as 'my_submission.csv'!")

Submission saved successfully as 'my_submission.csv'!
